# 12 · Declarative Pipelines (Lakeflow / DLT) + Data Quality

In notebooks 7–10 you built the medallion pipeline **imperatively** — you wrote
every read, transform, and write and managed order yourself. **Lakeflow
Declarative Pipelines** (formerly **Delta Live Tables / DLT**) let you instead
**declare the tables you want**, and Databricks figures out the dependencies,
runs them in order, handles incremental/streaming updates, retries, and — best of
all — **data-quality expectations**.

> **Important:** the `@dlt` code below defines a *pipeline*; it does **not** run in
> a normal interactive cell (`import dlt` only resolves inside a pipeline). You
> put this code in a notebook and attach it to a **Lakeflow Declarative Pipeline**
> (*Workflows → Pipelines → Create pipeline*). This notebook explains the pattern;
> the runnable cells at the end show the data-quality *idea* with plain Spark.

## 1 · The idea — declare, don't orchestrate

You write functions that **return a DataFrame**, decorated with `@dlt.table`. Each
function *is* a table. When one reads another via `dlt.read()` /
`dlt.read_stream()`, Lakeflow infers the **dependency graph** and builds them in
the right order — the whole Bronze→Silver→Gold DAG, managed for you.

Two kinds of dataset:
- **Streaming table** — incrementally processes new data (great for Bronze/Silver
  from Auto Loader).
- **Materialized view** — the full result, recomputed efficiently (great for Gold
  aggregates).

## 2 · A full medallion pipeline, declaratively

Below is the BrewBox medallion as a Lakeflow pipeline. Notice there's **no
orchestration code** — just table definitions. Copy this into a pipeline notebook
to run it.

In [ ]:
# === Lakeflow Declarative Pipeline (DLT) — runs in a PIPELINE, not this cell ===
import dlt
from pyspark.sql import functions as F

# --- BRONZE: stream raw order files with Auto Loader ---
@dlt.table(comment="Raw orders, ingested incrementally")
def orders_bronze():
    return (spark.readStream.format("cloudFiles")
            .option("cloudFiles.format", "json")
            .load("/Volumes/main/brewbox/landing/orders")
            .withColumn("_ingested_at", F.current_timestamp()))

# --- SILVER: clean + enforce data quality with EXPECTATIONS ---
@dlt.table(comment="Cleaned, validated orders")
@dlt.expect_or_drop("valid_order_id", "order_id IS NOT NULL")
@dlt.expect_or_drop("non_negative_amount", "amount >= 0")
@dlt.expect("known_status", "status IN ('completed','returned','cancelled')")
def orders_silver():
    return (dlt.read_stream("orders_bronze")
            .withColumn("order_ts", F.to_timestamp("order_ts"))
            .withColumn("status", F.lower(F.trim("status")))
            .dropDuplicates(["order_id"]))

# --- GOLD: aggregated mart (materialized view) ---
@dlt.table(comment="Daily completed revenue")
def daily_revenue_gold():
    return (dlt.read("orders_silver")
            .filter("status = 'completed'")
            .groupBy(F.to_date("order_ts").alias("date"))
            .agg(F.round(F.sum("amount"), 2).alias("revenue")))

## 3 · Data-quality **expectations** — the killer feature

Expectations are declarative data-quality rules attached to a table. Each has a
name + a boolean SQL condition, and a policy for what to do when a row fails:

| Decorator | On failure |
|---|---|
| `@dlt.expect("name", "cond")` | **keep** the bad row, but **record** the violation (metrics) |
| `@dlt.expect_or_drop("name", "cond")` | **drop** the bad row |
| `@dlt.expect_or_fail("name", "cond")` | **fail the pipeline** (stop everything) |

Lakeflow tracks pass/fail counts per expectation and shows them on the pipeline
graph — so data quality is **observable**, not hidden in asserts. You choose the
severity per rule (drop noise, but fail on a broken primary key).

## 4 · Why teams use it

- **Less code** — declare tables, skip the orchestration boilerplate.
- **Automatic dependencies & incremental** — the DAG and streaming state are managed.
- **Built-in data quality** — expectations with metrics and lineage.
- **Auto-recovery & retries** — the pipeline handles failures.
- **One definition, batch *or* streaming** — change the trigger, not the code.

**How to run it:** put the pipeline code in a notebook → *Workflows → Pipelines →
Create pipeline* → point it at the notebook → choose serverless → **Start**. The
pipeline UI shows the Bronze→Silver→Gold graph with row counts and expectation
results.

## 5 · The data-quality *idea*, runnable with plain Spark

You can't run `@dlt.expect` interactively, but here's the same concept by hand on
`brewbox.orders_silver` — count how many rows would pass/fail each rule. (In a
pipeline, Lakeflow does this automatically and enforces the policy.)

In [ ]:
try:
    spark
except NameError:
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()
from pyspark.sql import functions as F
spark.sql("USE SCHEMA brewbox")

df = spark.table("brewbox.orders_silver")
expectations = {
    "valid_order_id":      "order_id IS NOT NULL",
    "non_negative_amount": "amount >= 0",
    "known_status":        "status IN ('completed','returned','cancelled')",
}
total = df.count()
for name, cond in expectations.items():
    passed = df.filter(cond).count()
    print(f"{name:22s} pass={passed}/{total}  fail={total - passed}")

## 6 · Exercises

**Exercise 1 —** Add a plain-Spark check for a new expectation: `amount` should be
under 10000 (a sanity cap). How many rows fail?

In [ ]:
# Your turn (Exercise 1):

In [ ]:
# ✅ Solution 1
print("fail amount<10000:", spark.table("brewbox.orders_silver").filter("NOT (amount < 10000)").count())

**Exercise 2 —** Which expectation decorator would you use for a **primary-key**
rule you can never violate, and which for a **noisy optional field**? (Answer in
the cell.)

In [ ]:
print("Primary key -> @dlt.expect_or_fail (stop the pipeline; the data is broken).")
print("Noisy optional field -> @dlt.expect (keep rows, just record the metric) "
      "or @dlt.expect_or_drop if bad rows should be removed.")

## 7 · Recap & next

**Lakeflow Declarative Pipelines (DLT)** let you *declare* Bronze/Silver/Gold
tables and get dependency management, incremental/streaming updates, and — the
standout — **data-quality expectations** with built-in metrics. Far less code
than the imperative pipeline, and quality is observable.

**Next → `13` Orchestration:** schedule and coordinate jobs (Databricks Workflows
/ Lakeflow Jobs), the layer that runs your pipelines on time, in order, with
retries. 🚀